# Model comparison with cross-validation, then the threshold

One row is one telco customer; `Churn` is `Yes` when the customer left in the last month. The retention manager: "Each customer you flag gets a retention offer, which costs us 100 dollars whether they would have left or not. A customer who leaves without an offer costs us 400 dollars. Which model, and how many customers should we flag?"

## The starter

The cell repeats D01 5.6 and D02: `TotalCharges` converted to numbers, the `ColumnTransformer` `pre` (median and scaling for numbers, one-hot for text), a second one `tree_pre` with no scaling for the tree models, and the 80 / 20 split with `stratify=y, random_state=42`. Your 5,634 training customers and 1,409 test customers are the same as yesterday's.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
tree_pre = ColumnTransformer([          # the same blocks without the scaler: trees do not need it
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], verbose_feature_names_out=False).set_output(transform="pandas")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print(len(X_train), "training customers,", len(X_test), "test customers")

5634 training customers, 1409 test customers


## Step 1 · One set of folds for every model

`cv = StratifiedKFold(5, shuffle=True, random_state=42)`, created once and reused.

In [2]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=42)

## Step 2 · Cross-validate four models

For each model, run `cross_validate(model, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision", "neg_brier_score"], return_train_score=True)`. Every model sits inside its own pipeline, so every fold fits its own preprocessing.

- `make_pipeline(pre, DummyClassifier(strategy="prior"))`
- `make_pipeline(pre, LogisticRegression(max_iter=1000))`
- `make_pipeline(tree_pre, RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))`
- `make_pipeline(tree_pre, HistGradientBoostingClassifier(random_state=42))`

In [3]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

models = {
    "Dummy": make_pipeline(clone(pre), DummyClassifier(strategy="prior")),
    "Logistic": make_pipeline(clone(pre), LogisticRegression(max_iter=1000)),
    "Random forest": make_pipeline(clone(tree_pre), RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    "Boosted": make_pipeline(clone(tree_pre), HistGradientBoostingClassifier(random_state=42)),
}
cv_results = {name: cross_validate(m, X_train, y_train, cv=cv, return_train_score=True,
                                   scoring=["roc_auc", "average_precision", "neg_brier_score"])
              for name, m in models.items()}

## Step 3 · One table

Mean and standard deviation of the validation ROC AUC, mean average precision, mean Brier score (minus `test_neg_brier_score`), and the mean training ROC AUC.

In [4]:
table = pd.DataFrame({name: {"val_auc": f"{r['test_roc_auc'].mean():.3f} ± {r['test_roc_auc'].std():.3f}",
                             "avg_precision": round(r["test_average_precision"].mean(), 3),
                             "brier": round(-r["test_neg_brier_score"].mean(), 3),
                             "train_auc": round(r["train_roc_auc"].mean(), 3)}
                      for name, r in cv_results.items()}).T
table

,val_auc,avg_precision,brier,train_auc
Dummy,0.500 ± 0.000,0.265,0.195,0.5
Logistic,0.846 ± 0.013,0.661,0.135,0.85
Random forest,0.821 ± 0.012,0.613,0.146,1.0
Boosted,0.838 ± 0.007,0.65,0.14,0.963


## Step 4 · "Which model ranks customers best?"

Translate: the highest mean validation ROC AUC. Is the gap between the top two models larger than one standard deviation? Which model has the largest gap between training and validation AUC, and what does that gap mean (D02 3.1)?

In [5]:
auc = {name: r["test_roc_auc"].mean() for name, r in cv_results.items()}
gap_top2 = round(auc["Logistic"], 3) - round(auc["Boosted"], 3)     # the gap between the two table values
std_logistic = cv_results["Logistic"]["test_roc_auc"].std()
print(f"Logistic leads Boosted by {gap_top2:.3f}; its own standard deviation is {std_logistic:.3f}.")
for name, r in cv_results.items():
    print(f"{name:<14} train {r['train_roc_auc'].mean():.3f}  validation {r['test_roc_auc'].mean():.3f}")

Logistic leads Boosted by 0.008; its own standard deviation is 0.013.
Dummy          train 0.500  validation 0.500
Logistic       train 0.850  validation 0.846
Random forest  train 1.000  validation 0.821
Boosted        train 0.963  validation 0.838


## Step 5 · The cost as a function

`cost = 100 * FP + 400 * FN + 100 * TP`, from `confusion_matrix(y_true, y_pred).ravel()`. Wrap minus the mean cost per customer in `make_scorer`: the tuner maximizes its score, so a lower cost must be a higher score.

In [6]:
from sklearn.metrics import confusion_matrix, make_scorer


def total_cost(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return 100 * fp + 400 * fn + 100 * tp


def gain(y_true, y_pred):
    return -total_cost(y_true, y_pred) / len(y_true)


scorer = make_scorer(gain)

## Step 6 · Choose the threshold on the folds

`TunedThresholdClassifierCV(model, scoring=scorer, cv=cv, random_state=42).fit(X_train, y_train)`. Print `best_threshold_` and minus `best_score_` (the cost per customer on the folds).

In [7]:
from sklearn.model_selection import TunedThresholdClassifierCV

tuned = TunedThresholdClassifierCV(models["Logistic"], scoring=scorer, cv=cv, random_state=42).fit(X_train, y_train)
print(f"threshold {tuned.best_threshold_:.3f} | cost per customer on the folds {-tuned.best_score_:.2f}")

threshold 0.278 | cost per customer on the folds 61.68


## Step 7 · Only now, the test customers, once

The total cost at the 0.5 default (`model.predict`) and at your threshold (`tuned.predict`), and how many customers each flags.

In [8]:
default_model = clone(models["Logistic"]).fit(X_train, y_train)
pred_default, pred_tuned = default_model.predict(X_test), tuned.predict(X_test)
cost_default, cost_tuned = total_cost(y_test, pred_default), total_cost(y_test, pred_tuned)
print(f"default 0.5: {pred_default.sum()} flagged, total cost {cost_default:,}")
print(f"tuned:       {pred_tuned.sum()} flagged, total cost {cost_tuned:,}")

default 0.5: 318 flagged, total cost 97,800
tuned:       564 flagged, total cost 89,200


## Step 8 · "How many customers should we flag, and what does that save?"

Answer in two sentences with the numbers.

In [9]:
print(f"Flag {pred_tuned.sum()} of the {len(y_test):,} customers: every customer whose predicted churn probability "
      f"is at least {tuned.best_threshold_:.3f}. On the test customers that costs {cost_tuned:,} dollars, "
      f"{cost_default - cost_tuned:,} less than flagging at the 0.5 default.")

Flag 564 of the 1,409 customers: every customer whose predicted churn probability is at least 0.278. On the test customers that costs 89,200 dollars, 8,600 less than flagging at the 0.5 default.


## If you chose the boosted model

In [10]:
tuned_b = TunedThresholdClassifierCV(models["Boosted"], scoring=scorer, cv=cv, random_state=42).fit(X_train, y_train)
print(f"boosted: threshold {tuned_b.best_threshold_:.3f}, cost per customer on the folds {-tuned_b.best_score_:.2f}, "
      f"test cost {total_cost(y_test, tuned_b.predict(X_test)):,}")

boosted: threshold 0.209, cost per customer on the folds 62.68, test cost 88,600


## Check yourself

In [11]:
check = {}
for name, key in (("Dummy", "dummy"), ("Logistic", "logistic"), ("Random forest", "forest"), ("Boosted", "boosted")):
    r = cv_results[name]
    check[f"Table · {key} validation AUC"] = f"{r['test_roc_auc'].mean():.3f} ± {r['test_roc_auc'].std():.3f}"
    check[f"Table · {key} average precision"] = round(r["test_average_precision"].mean(), 3)
    check[f"Table · {key} Brier"] = round(-r["test_neg_brier_score"].mean(), 3)
    check[f"Table · {key} training AUC"] = round(r["train_roc_auc"].mean(), 3)
check.update({
    "Step 4 · logistic lead": round(gap_top2, 3),
    "Step 4 · logistic standard deviation": round(std_logistic, 3),
    "Step 6 · threshold": round(tuned.best_threshold_, 3),
    "Step 6 · cost per customer on the folds": round(-tuned.best_score_, 2),
    "Step 7 · flagged at 0.5": pred_default.sum(),
    "Step 7 · total cost at 0.5": cost_default,
    "Step 7 · flagged at the tuned threshold": pred_tuned.sum(),
    "Step 7 · total cost at the tuned threshold": cost_tuned,
    "Step 8 · saving": cost_default - cost_tuned,
    "Boosted · threshold": round(tuned_b.best_threshold_, 3),
    "Boosted · cost per customer on the folds": round(-tuned_b.best_score_, 2),
    "Boosted · test cost": total_cost(y_test, tuned_b.predict(X_test)),
})
for label, value in check.items():
    print(f"{label:<44} {value}")

Table · dummy validation AUC                 0.500 ± 0.000
Table · dummy average precision              0.265
Table · dummy Brier                          0.195
Table · dummy training AUC                   0.5
Table · logistic validation AUC              0.846 ± 0.013
Table · logistic average precision           0.661
Table · logistic Brier                       0.135
Table · logistic training AUC                0.85
Table · forest validation AUC                0.821 ± 0.012
Table · forest average precision             0.613
Table · forest Brier                         0.146
Table · forest training AUC                  1.0
Table · boosted validation AUC               0.838 ± 0.007
Table · boosted average precision            0.65
Table · boosted Brier                        0.14
Table · boosted training AUC                 0.963
Step 4 · logistic lead                       0.008
Step 4 · logistic standard deviation         0.013
Step 6 · threshold                           0.278
Step 6